# Lab 1 — Concepts, environnement & *function calling*
**Séance 1 · Concepts & architectures des agents · PGE5 / M2**

Ce premier TP pose les fondations sur lesquelles reposent tous les autres. On part
de l'appel de chat le plus simple et on construit, étape par étape, l'intuition de ce
qu'est un **agent**.

### Objectifs
1. Maîtriser l'**anatomie d'un appel LLM** : rôles, *system prompt*, température.
2. Lire le **coût** d'un appel (tokens entrée/sortie) — réflexe FinOps dès le départ.
3. Comprendre le ***function calling*** : décrire un outil, observer le modèle **décider**
   de l'appeler (ou non), exécuter l'outil, lui renvoyer le résultat.
4. Obtenir une **sortie structurée** (JSON) fiable.
5. Spécifier un agent avec la grille **PEAS** et la **taxonomie des environnements**.

> **Mode salle / hors-ligne.** Tout ce notebook fonctionne **sans clé d'API** : en
> l'absence de clé, `make_client` bascule sur un `MockLLMClient` piloté par un script.
> Dès que vous renseignez `.env`, les mêmes cellules appellent le vrai modèle.

> Prérequis : `pip install -r requirements.txt` puis (optionnel) un fichier `.env` — voir `README.md`.

## 0. Mise en route

On importe la **couche agnostique** `llm_helpers` et on détecte le mode (en ligne / hors-ligne).
Le helper `demo(script)` renvoie un **client neuf** à chaque appel : chaque cellule de démo
est ainsi **indépendante et ré-exécutable**, en ligne comme hors-ligne.

In [16]:
from llm_helpers import (
    make_client, credentials_available, LLMClient, MockLLMClient,
    ToolRegistry, tool_schema, run_agent, safe_calc,
)

ONLINE = credentials_available()
print("Mode :", "🌐 EN LIGNE (vrai modèle)" if ONLINE else "⚙️  HORS-LIGNE (mock)")

def demo(script=None):
    "Client frais pour une cellule de démo : vrai modèle si clé dispo, sinon mock scripté."
    return make_client(offline_script=script, quiet=True)

# Test de fumée
client = demo(["Un agent IA perçoit son environnement, raisonne, puis agit pour atteindre un but."])
rep = client.complete([{"role": "user", "content": "En une phrase : qu'est-ce qu'un agent IA ?"}])
print("\n→", rep.content)

Mode : 🌐 EN LIGNE (vrai modèle)

→ Un agent IA est un système autonome qui perçoit son environnement et agit intelligemment pour atteindre des objectifs spécifiques.


## 1. Anatomie d'un appel : rôles, *system prompt*, température

Un appel LLM = une **liste de messages**. Trois rôles principaux :

| Rôle | Rôle pédagogique |
|------|------------------|
| `system` | Cadre/persona/règles. **Fixé par le développeur**, jamais par l'utilisateur. |
| `user` | La requête. |
| `assistant` | La réponse du modèle (peut contenir des *tool calls*). |

La **température** contrôle l'aléa : `0` = déterministe (idéal pour les agents/outils),
`>0.7` = créatif. Pour un agent fiable, on travaille presque toujours à **température basse**.

In [17]:
# Même question, deux 'personas' via le system prompt
def ask(system, user, **kw):
    c = demo([f"[{system[:20]}…] réponse simulée"])
    return c.complete([{"role": "system", "content": system},
                       {"role": "user", "content": user}], **kw).content

print("PRUDENT :", ask("Tu es un juriste prudent. Réponds en nuançant.",
                       "Peut-on déployer un agent autonome en production ?"))
print("\nDIRECT  :", ask("Tu es un ingénieur direct. Réponds en une phrase tranchée.",
                        "Peut-on déployer un agent autonome en production ?"))

PRUDENT : Oui, il est *techniquement* de plus en plus envisageable de déployer un agent autonome en production. Cependant, d'un point de vue *juridique, éthique et

DIRECT  : Non, pas sans supervision humaine critique et des garde-fous infaillibles pour gérer son imprévisibilité.


### Lire le coût d'un appel

Chaque appel consomme des **tokens** (≈ ¾ d'un mot en français). Le client cumule l'usage et
estime un **coût** à partir d'une table de tarifs indicative. C'est la base du *FinOps* d'un agent.

In [18]:
c = demo(["Paris est la capitale de la France."])
c.complete([{"role": "user", "content": "Quelle est la capitale de la France ?"}])
print("Dernier appel :", c.last_usage)
print("Cumul         :", c.usage_report())
# En ligne : nombres réels du fournisseur. Hors-ligne : estimation simulée (coût ~0).

Dernier appel : {'input_tokens': 9, 'output_tokens': 9}
Cumul         : 1 appel(s) · 9 tok in / 9 tok out · ≈ $0.00003


## 2. Le *function calling* en pratique

Un LLM seul ne connaît pas l'heure, ne calcule pas de façon fiable et n'accède à aucune
donnée externe. On lui **décrit des outils** (nom, description, *JSON Schema* des arguments).
Le modèle choisit alors **quand** appeler un outil et **avec quels arguments** — c'est la
brique de base de **tout** agent.

Le cycle complet :

```
user → [modèle décide] → tool_call → [NOTRE code exécute] → tool_result → [modèle] → réponse
```

In [19]:
import datetime

def get_current_time(timezone: str = "Europe/Paris") -> str:
    "Renvoie l'heure courante (fuseau illustratif)."
    return f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ({timezone})"

time_tool = tool_schema(
    "get_current_time", "Donne la date et l'heure courantes.",
    {"timezone": {"type": "string", "description": "ex. Europe/Paris"}},
)

registry = ToolRegistry()
registry.register(time_tool, get_current_time)
print("Outils disponibles :", registry.names)

Outils disponibles : ['get_current_time']


In [20]:
# Question qui NÉCESSITE l'outil. On observe la DÉCISION du modèle (pas encore l'exécution).
client = demo([{"tool": "get_current_time", "arguments": {"timezone": "Europe/Paris"}}])
messages = [
    {"role": "system", "content": "Tu es un assistant. Utilise les outils quand c'est utile."},
    {"role": "user", "content": "Quelle heure est-il, précisément ?"},
]
reply = client.complete(messages, tools=registry.specs)
print("Texte           :", reply.content)
print("Appels d'outils :", reply.tool_calls)

Texte           : None
Appels d'outils : [{'id': 'function-call-9504699021203261184', 'name': 'get_current_time', 'arguments': {}}]


Le modèle **ne répond pas directement** : il renvoie un *tool call*. C'est à **notre code**
d'exécuter la fonction puis de renvoyer l'observation. La fonction `run_agent` automatise
cette boucle (et `verbose=True` en affiche chaque étape) :

In [21]:
client = demo([
    {"tool": "get_current_time", "arguments": {"timezone": "Europe/Paris"}},
    {"final": "Il est actuellement le créneau affiché ci-dessus ; minuit est dans quelques heures."},
])
messages = [
    {"role": "system", "content": "Tu es un assistant. Utilise les outils quand c'est utile."},
    {"role": "user", "content": "Quelle heure est-il, et dans combien de temps sera-t-il minuit ?"},
]
history = run_agent(client, registry, messages, verbose=True)

[étape 1] réponse finale → Je peux vous donner l'heure actuelle, mais je ne peux pas calculer le temps qu'il reste avant minuit. Voulez-vous que je vous donne l'heure actuelle ?


### Le modèle peut aussi décider de **ne pas** appeler d'outil

Donner des outils ne force pas leur usage. Pour une question conceptuelle, un bon modèle
répond directement. Savoir **quand ne pas agir** fait partie du comportement agentique.

In [22]:
client = demo([{"final": "Un agent est un système qui perçoit, décide et agit en boucle pour atteindre un but."}])
reply = client.complete(
    [{"role": "system", "content": "Réponds directement ; n'utilise un outil que si nécessaire."},
     {"role": "user", "content": "En une phrase, qu'est-ce qui distingue un agent d'un simple chatbot ?"}],
    tools=registry.specs,
)
print("Tool calls :", reply.tool_calls, "  (attendu : aucun)")
print("Réponse    :", reply.content)

Tool calls : []   (attendu : aucun)
Réponse    : Un agent peut effectuer des actions et prendre des décisions de manière autonome, tandis qu'un simple chatbot se limite à interagir en répondant à des questions ou en suivant des scripts prédéfinis.


## 3. Plusieurs outils — le modèle compose

On ajoute un second outil et on pose une question qui **enchaîne** deux outils.

In [23]:
def convert_currency(amount: float, rate: float) -> str:
    "Convertit un montant en le multipliant par un taux de change."
    return f"{amount * rate:.2f}"

registry.register(
    tool_schema("convert_currency", "Convertit un montant via un taux de change.",
                {"amount": {"type": "number"}, "rate": {"type": "number"}},
                ["amount", "rate"]),
    convert_currency,
)

client = demo([
    {"tool": "convert_currency", "arguments": {"amount": 250, "rate": 1.08}},
    {"final": "250 € ≈ 270.00 $ au taux 1,08."},
])
run_agent(client, registry,
          [{"role": "user", "content": "Convertis 250 euros en dollars au taux 1.08."}],
          verbose=True);

[étape 1] outil: convert_currency({'rate': 1.08, 'amount': 250})
          ↳ 270.00
[étape 2] réponse finale → 250 euros convertis en dollars au taux de 1.08 donnent 270.00 dollars.


## 4. Sortie structurée — du texte au **JSON fiable**

Le texte libre est pénible à parser de façon robuste. Pour brancher un LLM dans un système,
on veut une **sortie structurée** validée par un schéma. `complete_structured` force le modèle
à renvoyer un JSON conforme (technique agnostique : on déclare le schéma comme un outil et on
force son appel).

In [24]:
schema = {
    "type": "object",
    "properties": {
        "ville": {"type": "string"},
        "date": {"type": "string", "description": "AAAA-MM-JJ si présente, sinon ''"},
        "intention": {"type": "string", "enum": ["meteo", "reservation", "autre"]},
    },
    "required": ["ville", "intention"],
}

client = demo([{"tool": "emit",
                "arguments": {"ville": "Lyon", "date": "2026-06-22", "intention": "meteo"}}])
phrase = "Quel temps fera-t-il à Lyon le 22 juin 2026 ?"
result = client.complete_structured(
    [{"role": "user", "content": f"Extrait les informations de : '{phrase}'"}],
    schema,
)
print(result)
print("type :", type(result).__name__, "— directement exploitable en Python.")

{'date': '2026-06-22', 'ville': 'Lyon', 'intention': 'meteo'}
type : dict — directement exploitable en Python.


## 5. Spécifier un agent : grille **PEAS** & taxonomie des environnements

Avant de coder, on **spécifie**. La grille PEAS (Russell & Norvig) cadre la mission :

| Lettre | Question |
|--------|----------|
| **P**erformance | Comment mesure-t-on le succès ? |
| **E**nvironment | Dans quel monde l'agent opère-t-il ? |
| **A**ctuators | Quelles **actions** peut-il réaliser ? |
| **S**ensors | Quelles **entrées** perçoit-il ? |

On caractérise ensuite l'**environnement** :

| Propriété | Deux extrêmes |
|-----------|---------------|
| Observabilité | totalement observable ↔ partiellement observable |
| Déterminisme | déterministe ↔ stochastique |
| Épisodicité | épisodique ↔ séquentiel |
| Dynamique | statique ↔ dynamique |
| Agents | mono-agent ↔ multi-agents |

### ✍️ Exercice 5 — Complétez la grille PEAS

Pour un **agent assistant de révision** destiné à vos camarades de PGE5 (il répond aux
questions de cours, génère des QCM, planifie les révisions).

| Dimension | À compléter |
|-----------|-------------|
| **P**erformance | *…* |
| **E**nvironment | *…* |
| **A**ctuators | *…* |
| **S**ensors | *…* |

Puis classez l'environnement sur les 5 propriétés ci-dessus, **en justifiant**.

<details><summary>💡 Proposition de correction</summary>

| Dimension | Proposition |
|-----------|-------------|
| **P**erformance | Taux de bonnes réponses, pertinence des QCM, satisfaction étudiant, temps gagné. |
| **E**nvironment | Corpus de cours + historique de l'étudiant + questions en langage naturel. |
| **A**ctuators | Générer texte/QCM, planifier des sessions, citer des sources, poser des questions. |
| **S**ensors | Messages de l'étudiant, documents fournis, scores aux QCM passés. |

**Environnement** : *partiellement observable* (on ne connaît pas l'état mental de l'étudiant),
*stochastique* (réponses humaines imprévisibles), plutôt *séquentiel* (les révisions
s'enchaînent et dépendent du passé), *dynamique* (les connaissances évoluent entre sessions),
*mono-agent* (sauf si plusieurs étudiants/agents interagissent).
</details>

## 6. Exercices

> Convention du cours : tentez d'abord la cellule `# TODO`, puis dépliez la solution.

**Exercice 6.1 — Ajouter un outil et composer.**
Implémentez un outil `temperature_convert(celsius)` (→ Fahrenheit) et posez une question
qui combine `get_current_time` **et** ce nouvel outil.

In [ ]:
# TODO : implémenter temperature_convert, l'enregistrer, puis lancer run_agent.
def temperature_convert(celsius: float) -> str:
    ...  # remplacez par la conversion

# registry.register(tool_schema(...), temperature_convert)
# run_agent(demo([...]), registry, [{"role":"user","content":"..."}], verbose=True)

<details><summary>✅ Solution 6.1</summary>

```python
def temperature_convert(celsius: float) -> str:
    return f"{celsius * 9/5 + 32:.1f} °F"

registry.register(
    tool_schema("temperature_convert", "Convertit des °C en °F.",
                {"celsius": {"type": "number"}}, ["celsius"]),
    temperature_convert,
)

client = demo([
    {"tool": "get_current_time", "arguments": {"timezone": "Europe/Paris"}},
    {"tool": "temperature_convert", "arguments": {"celsius": 25}},
    {"final": "Il est l'heure indiquée ; 25 °C = 77.0 °F."},
])
run_agent(client, registry,
          [{"role": "user", "content": "Quelle heure est-il, et combien font 25°C en Fahrenheit ?"}],
          verbose=True)
```
</details>

**Exercice 6.2 — Extraction structurée.**
Définissez un schéma pour extraire d'un email `{expediteur, urgence(faible|moyenne|haute), action_requise}`
et testez-le sur une phrase de votre choix.

In [ ]:
# TODO : définir email_schema puis appeler complete_structured.
email_schema = {
    "type": "object",
    "properties": {
        # ... à compléter ...
    },
    "required": [],
}
# print(demo([{"tool":"emit","arguments":{...}}]).complete_structured([...], email_schema))

<details><summary>✅ Solution 6.2</summary>

```python
email_schema = {
    "type": "object",
    "properties": {
        "expediteur": {"type": "string"},
        "urgence": {"type": "string", "enum": ["faible", "moyenne", "haute"]},
        "action_requise": {"type": "string"},
    },
    "required": ["urgence", "action_requise"],
}
texte = "De: dsi@aivancity.fr — Le serveur de prod est down, merci d'intervenir immédiatement."
client = demo([{"tool": "emit", "arguments": {
    "expediteur": "dsi@aivancity.fr", "urgence": "haute",
    "action_requise": "Rétablir le serveur de production"}}])
print(client.complete_structured(
    [{"role": "user", "content": f"Extrait les infos de cet email : {texte}"}], email_schema))
```
</details>

## ✅ Récapitulatif

- Un appel LLM = des **messages** (`system`/`user`/`assistant`) + une **température**.
- On **mesure** tokens et coût dès le premier appel.
- Le ***function calling*** laisse le modèle **décider** d'agir ; **notre code** exécute.
- La **sortie structurée** rend les réponses exploitables par un programme.
- **PEAS** + taxonomie d'environnement = la fiche d'identité d'un agent.

➡️ **Lab 2** : on assemble ces briques en une vraie **boucle ReAct**, on ajoute **mémoire**,
**garde-fous** et **auto-critique** — un agent codé entièrement à la main.